# Encoding shared_av: gradient, myelin and network alignment (59k)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from shared.paths import DATA, OUTPUTS

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import re
import subprocess
from scipy.stats import spearmanr

OUTPUTS_BASE   = str(OUTPUTS)
DATA_BASE      = str(DATA)
HCP_DIR        = f'{DATA_BASE}/HCP_S1200_GroupAvg_v1'
TEMPLATE_CIFTI = f'{DATA_BASE}/preprocessed/average_sub/raw/group_average_raw_cortex_59k.dtseries.nii'
MYELIN_CIFTI   = f'{DATA_BASE}/HCP_S1200_GroupAvg_v1/S1200.SmoothedMyelinMap_BC_MSMAll.32k_fs_LR.dscalar.nii'
GRAD_DIR       = f'{DATA_BASE}/margulies2016'
WB             = '/opt/workbench/bin_linux64/wb_command'

SPHERE_32K = {
    'L': f'{HCP_DIR}/S1200.L.sphere.32k_fs_LR.surf.gii',
    'R': f'{HCP_DIR}/S1200.R.sphere.32k_fs_LR.surf.gii',
}
SPHERE_59K = {
    'L': f'{HCP_DIR}/L.sphere.59k_fs_LR.surf.gii',
    'R': f'{HCP_DIR}/R.sphere.59k_fs_LR.surf.gii',
}

GRADIENT_LABELS = {
    1:  'G1: Unimodal→Heteromodal',
    2:  'G2: Visual→Somatomotor',
    3:  'G3: DMN→FPN',
    4:  'G4: Dorsal→Ventral Attn',
    5:  'G5: Visual→Ventral Attn',
    6:  'G6: Default→Dorsal Attn',
    7:  'G7: Med.Par.→Lat.PFC',
    8:  'G8: Primary→Premotor',
    9:  'G9: Foveal→Peripheral',
    10: 'G10: Auditory→Somatosens.',
}

bm_axis = nib.load(TEMPLATE_CIFTI).header.get_axis(1)
left_cifti_idx = right_cifti_idx = None
for name, _, model in bm_axis.iter_structures():
    if 'CORTEX_LEFT'  in name: left_cifti_idx  = model.vertex
    if 'CORTEX_RIGHT' in name: right_cifti_idx = model.vertex


CACHE_DIR = Path('/tmp/gradient_59k_cache')
CACHE_DIR.mkdir(exist_ok=True)

FIGURE_DIR = Path(f'{OUTPUTS_BASE}/figures/gradient_analysis')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def _resample_to_59k(src_path, hemi, out_path):
    subprocess.run(
        [WB, '-metric-resample', str(src_path),
         SPHERE_32K[hemi], SPHERE_59K[hemi],
         'BARYCENTRIC', str(out_path)],
        check=True, capture_output=True
    )

def load_gradient_59k(grad_num):
    grad_str = f'0{grad_num}' if grad_num < 10 else '10'
    parts = []
    for hemi, cidx in [('L', left_cifti_idx), ('R', right_cifti_idx)]:
        cached = CACHE_DIR / f'grad{grad_str}_59k_{hemi}.func.gii'
        if not cached.exists():
            src = (f'{GRAD_DIR}/fcgradient{grad_str}/fsLR'
                   f'/source-margulies2016_desc-fcgradient{grad_str}'
                   f'_space-fsLR_den-32k_hemi-{hemi}_feature.func.gii')
            _resample_to_59k(src, hemi, cached)
        parts.append(nib.load(str(cached)).agg_data()[cidx])
    return np.concatenate(parts)

def load_myelin_59k():
    parts = []
    myelin_img  = nib.load(MYELIN_CIFTI)
    myelin_bm32 = myelin_img.header.get_axis(1)
    myelin_data = myelin_img.get_fdata().squeeze()
    for hemi, cidx in [('L', left_cifti_idx), ('R', right_cifti_idx)]:
        cached_59k = CACHE_DIR / f'myelin_59k_{hemi}.func.gii'
        if not cached_59k.exists():
            tmp_32k = CACHE_DIR / f'myelin_32k_{hemi}.func.gii'
            full_arr = np.full(32492, np.nan, dtype=np.float32)
            for name, slc, mdl in myelin_bm32.iter_structures():
                if (hemi == 'L' and 'LEFT' in name) or (hemi == 'R' and 'RIGHT' in name):
                    full_arr[mdl.vertex] = myelin_data[slc].astype(np.float32)
                    break
            darray = nib.gifti.GiftiDataArray(
                data=full_arr,
                intent=nib.nifti1.intent_codes['NIFTI_INTENT_NONE'],
                datatype='NIFTI_TYPE_FLOAT32'
            )
            nib.save(nib.gifti.GiftiImage(darrays=[darray]), str(tmp_32k))
            _resample_to_59k(tmp_32k, hemi, cached_59k)
        parts.append(nib.load(str(cached_59k)).agg_data()[cidx])
    return np.concatenate(parts)

def _hexbin_scatter(ax, x, y, xlabel, ylabel, title):
    valid = ~np.isnan(x) & ~np.isnan(y)
    xv, yv = x[valid], y[valid]
    hb = ax.hexbin(xv, yv, gridsize=60, cmap='viridis', mincnt=1, linewidths=0.1)
    m, b = np.polyfit(xv, yv, 1)
    xline = np.linspace(xv.min(), xv.max(), 300)
    ax.plot(xline, m * xline + b, color='red', linewidth=1.5, alpha=0.9)
    r, _ = spearmanr(xv, yv)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'{title}  (ρ = {r:.3f})', fontsize=10)
    return hb

## Encoding: shared audio-video variance against gradients and myelin

In [ ]:
import pandas as pd

from encoding.shared.encoding_utils import load_cifti_maps
from encoding.shared.layout import fit_folder, group_dir
from encoding.shared.tags import own_tag, pair_tag

ENC_FIGURE_DIR = FIGURE_DIR / 'encoding_shared_av'
ENC_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
ENC_PAIRS = [own_tag('pe-av-small-16-frame'), pair_tag('whisper-large-v3-l30', 'qwen2.5-vl-3b-vit-l32'),
             pair_tag('clap-larger', 'viclip-l14'), pair_tag('dasheng-1.2b', 'videomaev2-giant-d50')]

enc_path = fit_folder(group_dir('cortex'), 'pe-av-small-16-frame', 5) / 'shared_av_zscore.dscalar.nii'
assert (nib.load(str(enc_path)).header.get_axis(1).vertex == bm_axis.vertex).all()
shared_av = load_cifti_maps(str(enc_path))
enc_gradients = {g: load_gradient_59k(g) for g in range(1, 11)}
enc_targets = {**{GRADIENT_LABELS[g]: enc_gradients[g] for g in range(1, 11)}, 'T1w/T2w myelin': load_myelin_59k()}

rows = []
for pair in ENC_PAIRS:
    for target_label, target in enc_targets.items():
        valid = np.isfinite(shared_av[pair]) & np.isfinite(target)
        rho, _ = spearmanr(target[valid], shared_av[pair][valid])
        rows.append((pair, target_label, rho, int(valid.sum())))
enc_table = pd.DataFrame(rows, columns=['pair', 'map', 'spearman_rho', 'n_grayordinates'])
enc_table.pivot(index='pair', columns='map', values='spearman_rho').loc[ENC_PAIRS, list(enc_targets)].round(3)

In [ ]:
print(enc_table.groupby('map', sort=False).n_grayordinates.agg(['min', 'max']))

fig, ax = plt.subplots(figsize=(13, 5))
width = 0.8 / len(ENC_PAIRS)
colors = plt.cm.tab10(np.arange(len(ENC_PAIRS)))
for i, pair in enumerate(ENC_PAIRS):
    values = enc_table[enc_table.pair == pair].set_index('map').loc[list(enc_targets), 'spearman_rho']
    ax.bar(np.arange(len(enc_targets)) + (i - (len(ENC_PAIRS) - 1) / 2) * width, values, width, label=pair, color=colors[i])
ax.set_xticks(np.arange(len(enc_targets)))
ax.set_xticklabels(list(enc_targets), rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Spearman ρ')
ax.set_title('shared_av (encoding, 5 s) vs Margulies 2016 gradients and myelin')
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.legend(fontsize=8, ncol=2, loc='upper right')
plt.tight_layout()
fig.savefig(ENC_FIGURE_DIR / 'shared_av_gradient_myelin_bars.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
PEAV_PAIR = ENC_PAIRS[0]
grad1 = enc_gradients[1]
myelin = enc_targets['T1w/T2w myelin']

fig, ax = plt.subplots(figsize=(6, 5))
hb = _hexbin_scatter(ax, grad1, shared_av[PEAV_PAIR], GRADIENT_LABELS[1], 'shared_av (R²)', f'{PEAV_PAIR}  (n = {np.isfinite(grad1).sum():,})')
plt.colorbar(hb, ax=ax, label='vertex count')
plt.tight_layout()
fig.savefig(ENC_FIGURE_DIR / f'gradient1_hexbin_shared_av_{PEAV_PAIR}.png', dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, len(ENC_PAIRS), figsize=(5 * len(ENC_PAIRS), 8.5), sharex='row')
for column, pair in enumerate(ENC_PAIRS):
    for row, (target, xlabel) in enumerate(((grad1, 'G1'), (myelin, 'T1w/T2w myelin'))):
        _hexbin_scatter(axes[row, column], target, shared_av[pair], xlabel, 'shared_av (R²)', pair)
        axes[row, column].title.set_fontsize(8)
fig.suptitle(f'shared_av vs {GRADIENT_LABELS[1]} (top) and myelin (bottom), one panel per pair', fontsize=12)
plt.tight_layout()
fig.savefig(ENC_FIGURE_DIR / 'gradient1_myelin_hexbin_shared_av_pairs.png', dpi=300, bbox_inches='tight')
plt.show()

## shared_av by Yeo 7 network

In [ ]:
YEO_PAIR = own_tag('pe-av-small-16-frame')
YEO_DIR = DATA / 'HCP_S1200_GroupAvg_v1/manual 59k/RSN_resampled'
YEO_MAP = '7 RSN Networks (YKS11 - Yeo et al., JNP, 2011)'
YEO_NAMES = {1: 'Visual', 2: 'Somatomotor', 3: 'Dorsal attention', 4: 'Ventral attention', 5: 'Limbic', 6: 'Frontoparietal', 7: 'Default'}

network, colors = [], {}
for hemi, vertices in (('LEFT', left_cifti_idx), ('RIGHT', right_cifti_idx)):
    gifti = nib.load(str(YEO_DIR / f'RSN-networks_59k_{hemi}.label.gii'))
    darray = next(d for d in gifti.darrays if d.meta['Name'] == YEO_MAP)
    key_to_network, key_to_color = {}, {}
    for label in gifti.labeltable.labels:
        found = re.fullmatch(r'7Networks_(\d)', label.label)
        if found:
            key_to_network[label.key] = int(found[1])
            colors.setdefault(int(found[1]), (label.red, label.green, label.blue))
            assert colors[int(found[1])] == (label.red, label.green, label.blue)
    network.append(np.array([key_to_network.get(k, 0) for k in darray.data[vertices]]))
network = np.concatenate(network)
assert len(network) == len(shared_av[YEO_PAIR])

rows = []
for number, name in YEO_NAMES.items():
    values = shared_av[YEO_PAIR][(network == number) & np.isfinite(shared_av[YEO_PAIR])]
    rows.append((name, len(values), values.mean(), np.median(values)))
yeo_table = pd.DataFrame(rows, columns=['network', 'n_grayordinates', 'mean_shared_av', 'median_shared_av'])
print(yeo_table.to_string(index=False, float_format=lambda v: f'{v:.5f}'))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(yeo_table.network, yeo_table.mean_shared_av, color=[colors[n] for n in YEO_NAMES])
ax.set_ylabel('mean shared_av (R²)')
ax.set_title(f'shared_av by Yeo 7 network, {YEO_PAIR} (5 s, loro, z-scored, cortex)')
ax.axhline(0, color='k', linewidth=0.8)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
fig.savefig(ENC_FIGURE_DIR / f'shared_av_yeo7_{YEO_PAIR}.png', dpi=600, bbox_inches='tight')
plt.show()